# Taller - Operaciones con Tablas de Datos


## Fase 1 - Exploración y diagnóstico

Primero cargo el archivo y reviso cómo vienen los datos

In [0]:
import pandas as pd

df = pd.read_csv("/Workspace/Users/nicold.ruizd@uqvirtual.edu.co/Drafts/caso_practico_ventas (1).csv")
print("Shape:", df.shape)
df.head(10)

In [0]:
df.tail(5)

`fecha` quedó como texto en vez de fecha, arreglo (Fase 3)

In [0]:
df.dtypes

In [0]:
nulos = df.isnull().sum()
pct = (nulos/len(df)*100).round(2)
pd.DataFrame({"nulos": nulos, "pct_%": pct})

In [0]:
df[['cantidad','precio_unitario']].describe()

`precio_unitario` tiene una desviación estándar de 924, muchísimo más alta que su media de 186, y el máximo da 9999 que no tiene sentido para ningún producto del catálogo. Lo corrijo en la Fase 2

In [0]:
print("Duplicados exactos:", df.duplicated().sum())
print("Duplicados por id_transaccion:", df.duplicated(subset=['id_transaccion']).sum())

## Fase 2 - Limpieza

Con los problemas ya identificados en la Fase 1 se corrigen

In [0]:
antes = len(df)
df = df.drop_duplicates(subset=['id_transaccion']).reset_index(drop=True)
print(f"Filas antes: {antes} -> despues: {len(df)} (se eliminaron {antes-len(df)})")

In [0]:
df['metodo_pago'] = df['metodo_pago'].replace({'Tarjeta Credito': 'Tarjeta de Crédito'})
df['metodo_pago'].value_counts()

In [0]:
mediana_cant = df['cantidad'].median()
df['cantidad'] = df['cantidad'].fillna(mediana_cant)

moda_sat = df['satisfaccion_cliente'].mode()[0]
df['satisfaccion_cliente'] = df['satisfaccion_cliente'].fillna(moda_sat)

print("Mediana cantidad usada:", mediana_cant)
print("Moda satisfaccion usada:", moda_sat)

Ahora el outlier, es la fila TXN-10060, de la categoría Ropa, con precio_unitario de 9999  un error de digitación, porque ninguna prenda cuesta eso, en vez de borrar toda la fila (el resto de datos de esa están bien), solo cambio el precio por el promedio de Ropa, sin contar ese valor raro en el cálculo

In [0]:
precio_medio_ropa = df[(df['categoria_producto']=='Ropa') & (df['precio_unitario']!=9999.0)]['precio_unitario'].mean()
print(f"Precio medio de Ropa (sin outlier): {precio_medio_ropa:.2f}")

df.loc[df['precio_unitario']==9999.0, 'precio_unitario'] = precio_medio_ropa

A precio_unitario todavía le quedaban algunos nulos. Los llené con la mediana de precio de cada categoría, para no dañar el promedio con el outlier que ya corregí arriba

In [0]:
df['precio_unitario'] = df.groupby('categoria_producto')['precio_unitario'].transform(
    lambda s: s.fillna(s.median())
)
print("Nulos restantes en todo el dataframe:")
df.isnull().sum()

## Fase 3 - Transformación

Con los datos ya limpios, creo las columnas nuevas que voy a necesitar para el análisis

Primero el monto total por transacción, que es cantidad por precio

In [0]:
df['monto_total'] = df['cantidad'] * df['precio_unitario']
df[['id_transaccion','cantidad','precio_unitario','monto_total']].head()

convierto fecha a un tipo de dato real y saco el día de la semana y el mes, por si después sirven para ver patrones

In [0]:
df['fecha'] = pd.to_datetime(df['fecha'])
df['dia_semana'] = df['fecha'].dt.day_name()
df['mes'] = df['fecha'].dt.month
df[['fecha','dia_semana','mes']].head()

Por último clasifico cada venta en Bajo, Medio o Alto según el monto, menos de 100, entre 100 y 500, más de 500

In [0]:
def clasificar(m):
    if m < 100:
        return 'Bajo'
    elif m <= 500:
        return 'Medio'
    else:
        return 'Alto'

df['rango_monto'] = df['monto_total'].apply(clasificar)
df['rango_monto'].value_counts()

## Fase 4 - Agregación

Ahora agrupo los datos para sacar conclusiones

¿Qué categoría deja más plata y cuál vende más unidades en promedio?

In [0]:
resumen_cat = df.groupby('categoria_producto').agg(
    ingresos_totales=('monto_total','sum'),
    promedio_unidades=('cantidad','mean')
).round(2).sort_values('ingresos_totales', ascending=False)
resumen_cat

Hogar es la que más ingresos genera, seguida de Ropa, ahora cruzo región con categoría para ver dónde se vende más de cada cosa

In [0]:
pivot = pd.pivot_table(df, values='monto_total', index='region', columns='categoria_producto', aggfunc='sum', fill_value=0).round(2)
pivot

Por último reviso si el método de pago tiene alguna relación con qué tan satisfecho queda el cliente

In [0]:
df.groupby('metodo_pago')['satisfaccion_cliente'].mean().round(2).sort_values(ascending=False)

Curioso: efectivo es el que deja clientes más contentos y tarjeta de crédito el que menos. Puede ser casualidad por el tamaño de la muestra, pero vale la pena mencionarlo

## Fase 5 - Filtrado y muestreo

Filtro las ventas de la región Norte que tuvieron buena calificación 4 o 5

In [0]:
filtro = df[(df['region']=='Norte') & (df['satisfaccion_cliente']>=4)]
print(f"Filas que cumplen: {len(filtro)} de {len(df)}")
filtro[['id_transaccion','region','satisfaccion_cliente','categoria_producto','monto_total']]

In [0]:
muestra = pd.concat([
    g.sample(frac=0.10, random_state=42) for _, g in df.groupby('region')
])
print(f"Tamaño de la muestra: {len(muestra)} de {len(df)} filas ({len(muestra)/len(df)*100:.1f}%)")
muestra['region'].value_counts()

Me dio 11 filas (9.2%) y no 12 exactas (10%) paso porque al sacar el 10% por región, algunas dan números con decimales 

In [0]:
df.to_csv("ventas_limpio_final.csv", index=False)
print("Dataset final guardado:", df.shape)